# 📈 Análisis de Series Temporales

**Módulo 03** | **Sesión 6** | **Duración estimada: 1h 30min**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gonzalezulises/formacion-docente-bi-faces/blob/main/modulo-03-modelado-predictivo/notebooks/03_06_series_temporales.ipynb)

## 🎯 Objetivos de Aprendizaje

Al finalizar esta sesión, serás capaz de:

| # | Competencia | Nivel |
|---|-------------|-------|
| 1 | Identificar y describir los componentes de una serie temporal (tendencia, estacionalidad, ciclo, residuo) | Comprensión |
| 2 | Descomponer una serie temporal usando Python y visualizar sus componentes | Aplicación |
| 3 | Calcular estadísticas descriptivas temporales (medias móviles, desviación estándar móvil) | Aplicación |
| 4 | Evaluar la estacionariedad de una serie mediante la prueba ADF | Análisis |
| 5 | Ajustar un modelo ARIMA básico y generar pronósticos con intervalos de confianza | Aplicación |
| 6 | Aplicar el flujo completo de análisis temporal a indicadores económicos venezolanos | Síntesis |

## 💡 ¿Por qué es importante para tu práctica docente?

Como profesores de FACES-UC en áreas como Economía, Administración, Contaduría y Relaciones Industriales, trabajamos constantemente con datos que evolucionan en el tiempo:

- **Economía:** inflación mensual, tipo de cambio, reservas internacionales, PIB trimestral
- **Administración:** ventas mensuales, presupuesto ejecutado por período, indicadores de gestión
- **Contaduría:** flujo de caja mensual, ingresos y gastos periódicos, tendencias financieras
- **Relaciones Industriales:** rotación de personal trimestral, evolución salarial, índices de ausentismo

El análisis de series temporales nos permite **entender patrones históricos** y **proyectar valores futuros** — herramientas fundamentales para la toma de decisiones informada. Esta sesión te dará las técnicas básicas para analizar cualquier serie temporal con Python, desde la exploración visual hasta la generación de pronósticos con ARIMA.

> **Nota pedagógica:** No necesitas ser estadístico para usar estas herramientas. Nos enfocaremos en la **intuición** y la **interpretación** de resultados, no en las demostraciones matemáticas.

In [ ]:
# ============================================================
# Configuración inicial - Ejecutar esta celda primero
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
import warnings

# Suprimir advertencias que no son relevantes para el aprendizaje
warnings.filterwarnings('ignore')

# Configuración visual
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

print('✅ Bibliotecas cargadas correctamente')
print(f'   pandas {pd.__version__}')
print(f'   numpy {np.__version__}')

---

## 📊 1. ¿Qué es una serie temporal?

Una **serie temporal** (o serie de tiempo) es una secuencia de observaciones medidas en **intervalos regulares** a lo largo del tiempo.

### Ejemplos cotidianos en nuestras áreas:

| Área | Ejemplo de serie temporal | Frecuencia típica |
|------|--------------------------|--------------------|
| Economía | Inflación mensual de Venezuela | Mensual |
| Economía | PIB trimestral | Trimestral |
| Administración | Ventas de una empresa | Diario / Mensual |
| Contaduría | Presupuesto ejecutado por la facultad | Mensual |
| RRII | Tasa de rotación de personal | Trimestral |

### ¿En qué se diferencia de datos de corte transversal?

- **Corte transversal:** muchas observaciones en un **mismo momento** (ej: notas de 3,000 estudiantes en un semestre)
- **Serie temporal:** una variable medida en **muchos momentos** (ej: inflación mensual de 2015 a 2025)

La diferencia fundamental es que en una serie temporal **el orden importa** — no podemos reorganizar los datos aleatoriamente sin perder información.

In [ ]:
# ============================================================
# Cargar datos de indicadores del BCV
# ============================================================
indicadores = pd.read_csv(
    '../../datasets/economia/indicadores_bcv.csv',
    parse_dates=['fecha']  # Convertir la columna 'fecha' a datetime automáticamente
)

# Establecer la fecha como índice (fundamental para series temporales)
indicadores = indicadores.set_index('fecha')

# Explorar la estructura de los datos
print('=== Indicadores Económicos del BCV ===')
print(f'Período: {indicadores.index.min().strftime("%B %Y")} — {indicadores.index.max().strftime("%B %Y")}')
print(f'Observaciones: {len(indicadores)}')
print(f'Frecuencia: Mensual')
print(f'\nColumnas disponibles:')
for col in indicadores.columns:
    no_nulos = indicadores[col].notna().sum()
    print(f'  • {col}: {no_nulos} valores ({no_nulos/len(indicadores)*100:.0f}%)')

print(f'\n--- Primeras 5 filas ---')
display(indicadores.head())

In [ ]:
# ============================================================
# Visualizar la inflación mensual como serie temporal
# ============================================================
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(indicadores.index, indicadores['inflacion_mensual'], 
        color='#e74c3c', linewidth=1.2, alpha=0.9)
ax.fill_between(indicadores.index, indicadores['inflacion_mensual'], 
                alpha=0.15, color='#e74c3c')

ax.set_title('Inflación Mensual de Venezuela (2015–2025)', fontsize=14, fontweight='bold')
ax.set_xlabel('Fecha')
ax.set_ylabel('Inflación mensual (%)')
ax.axhline(y=indicadores['inflacion_mensual'].mean(), color='gray', 
           linestyle='--', alpha=0.7, label=f'Promedio: {indicadores["inflacion_mensual"].mean():.1f}%')
ax.legend()

plt.tight_layout()
plt.show()

print('\n🔍 Observa en el gráfico:')
print('   - ¿Hay una tendencia general (subida o bajada)?')
print('   - ¿Se repiten patrones en ciertos meses del año (estacionalidad)?')
print('   - ¿Hay picos inusuales (valores atípicos)?')

---

## 🔄 2. Componentes de una serie temporal

Toda serie temporal puede descomponerse conceptualmente en **cuatro componentes:**

| Componente | ¿Qué captura? | Ejemplo |
|------------|---------------|----------|
| **Tendencia** | Dirección general a largo plazo | La inflación tiende a subir o bajar |
| **Estacionalidad** | Patrones que se repiten en períodos fijos | Gastos universitarios suben en enero y septiembre (inicio de semestre) |
| **Ciclo** | Oscilaciones de largo plazo no regulares | Ciclos económicos de expansión y recesión |
| **Residuo (ruido)** | Variación aleatoria no explicada | Fluctuaciones impredecibles mes a mes |

Pensemos en una analogía: es como analizar el nivel de un río:
- La **tendencia** sería el aumento gradual por el deshielo de un glaciar
- La **estacionalidad** serían las crecidas predecibles en temporada de lluvias
- El **residuo** sería la variación diaria impredecible

In [ ]:
# ============================================================
# Cargar datos de presupuesto de la facultad
# ============================================================
presupuesto = pd.read_csv('../../datasets/universidad/presupuesto_facultad.csv')

print('=== Presupuesto de la Facultad ===')
print(f'Columnas: {list(presupuesto.columns)}')
print(f'Partidas: {presupuesto["partida"].unique()}')
print(f'Período: {presupuesto["anio"].min()} — {presupuesto["anio"].max()}')
print(f'\nPrimeras filas:')
display(presupuesto.head(10))

In [ ]:
# ============================================================
# Agregar el presupuesto por mes (todas las partidas)
# ============================================================

# Crear una fecha a partir de año y mes
presupuesto['fecha'] = pd.to_datetime(
    presupuesto['anio'].astype(str) + '-' + presupuesto['mes'].astype(str) + '-01'
)

# Sumar el monto ejecutado de TODAS las partidas por mes
ejecutado_mensual = presupuesto.groupby('fecha')['ejecutado'].sum().sort_index()

# Asignar frecuencia mensual explícita al índice
ejecutado_mensual.index = pd.DatetimeIndex(ejecutado_mensual.index, freq='MS')

print(f'Serie temporal de presupuesto ejecutado total:')
print(f'  Observaciones: {len(ejecutado_mensual)}')
print(f'  Período: {ejecutado_mensual.index[0].strftime("%Y-%m")} a {ejecutado_mensual.index[-1].strftime("%Y-%m")}')
print(f'  Promedio mensual: ${ejecutado_mensual.mean():,.0f}')

# Visualizar
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(ejecutado_mensual.index, ejecutado_mensual.values, 
        color='#2980b9', linewidth=1.3)
ax.fill_between(ejecutado_mensual.index, ejecutado_mensual.values, alpha=0.1, color='#2980b9')
ax.set_title('Presupuesto Ejecutado Total por Mes — FACES-UC', fontsize=14, fontweight='bold')
ax.set_xlabel('Fecha')
ax.set_ylabel('Monto ejecutado ($)')
plt.tight_layout()
plt.show()

---

## 🔬 3. Descomposición de una serie temporal

La **descomposición** es una técnica que separa automáticamente una serie en sus componentes. Python nos permite hacerlo con una sola línea de código.

Hay dos tipos de descomposición:

- **Aditiva:** `Y(t) = Tendencia + Estacionalidad + Residuo`  
  → Usar cuando la amplitud de las fluctuaciones estacionales es **constante** en el tiempo

- **Multiplicativa:** `Y(t) = Tendencia × Estacionalidad × Residuo`  
  → Usar cuando las fluctuaciones crecen **proporcionalmente** con la tendencia

Para nuestro presupuesto universitario, usaremos el modelo **aditivo** como punto de partida.

In [ ]:
# ============================================================
# Descomposición de la serie de presupuesto ejecutado
# ============================================================

# Descomponer usando modelo aditivo con período de 12 meses (anual)
descomposicion = seasonal_decompose(
    ejecutado_mensual, 
    model='additive',  # Modelo aditivo
    period=12           # Período estacional = 12 meses (1 año)
)

# Graficar los 4 componentes
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

# Serie original
axes[0].plot(descomposicion.observed, color='#2c3e50', linewidth=1.2)
axes[0].set_title('Serie Original (Presupuesto Ejecutado)', fontweight='bold')
axes[0].set_ylabel('$')

# Tendencia
axes[1].plot(descomposicion.trend, color='#e74c3c', linewidth=2)
axes[1].set_title('Tendencia', fontweight='bold')
axes[1].set_ylabel('$')

# Estacionalidad
axes[2].plot(descomposicion.seasonal, color='#27ae60', linewidth=1.2)
axes[2].set_title('Estacionalidad (patrón que se repite cada año)', fontweight='bold')
axes[2].set_ylabel('$')

# Residuo
axes[3].plot(descomposicion.resid, color='#8e44ad', linewidth=0.8, alpha=0.7)
axes[3].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[3].set_title('Residuo (variación no explicada)', fontweight='bold')
axes[3].set_ylabel('$')
axes[3].set_xlabel('Fecha')

plt.suptitle('Descomposición del Presupuesto Ejecutado — FACES-UC', 
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('\n🔍 Interpretación:')
print('   • TENDENCIA: Muestra la dirección general del presupuesto a largo plazo.')
print('   • ESTACIONALIDAD: Revela los meses donde sistemáticamente se gasta más o menos.')
print('   • RESIDUO: Lo que queda — fluctuaciones aleatorias o eventos inusuales.')
print('\n   Si los residuos son grandes respecto a la serie, el modelo no captura bien los datos.')

---

## 📐 4. Estadísticas descriptivas temporales

En series temporales, las estadísticas clásicas (media, desviación estándar) no son muy informativas porque resumen **todo el período** en un solo número. Necesitamos estadísticas que **evolucionen con el tiempo:**

| Estadística | ¿Qué hace? | ¿Para qué sirve? |
|-------------|-----------|-------------------|
| **Media móvil** (rolling mean) | Promedio de los últimos N períodos | Suavizar fluctuaciones y ver la tendencia |
| **Desviación estándar móvil** (rolling std) | Volatilidad de los últimos N períodos | Detectar períodos de mayor incertidumbre |
| **Media acumulada** (expanding mean) | Promedio desde el inicio hasta cada punto | Ver cómo evoluciona el promedio histórico |

La ventana (N) determina cuánto se "suaviza" la serie:
- **Ventana corta (3 meses):** Captura cambios recientes, pero sigue siendo ruidosa
- **Ventana larga (12 meses):** Muy suave, pero puede ocultar cambios recientes

In [ ]:
# ============================================================
# Medias móviles sobre la inflación mensual
# ============================================================

inflacion = indicadores['inflacion_mensual'].copy()

# Calcular medias móviles de 3 y 12 meses
media_movil_3 = inflacion.rolling(window=3).mean()   # Promedio de últimos 3 meses
media_movil_12 = inflacion.rolling(window=12).mean() # Promedio de últimos 12 meses

# Visualizar
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(inflacion.index, inflacion, color='#bdc3c7', linewidth=0.8, 
        alpha=0.7, label='Inflación mensual (original)')
ax.plot(media_movil_3.index, media_movil_3, color='#e67e22', 
        linewidth=1.5, label='Media móvil 3 meses')
ax.plot(media_movil_12.index, media_movil_12, color='#c0392b', 
        linewidth=2.5, label='Media móvil 12 meses')

ax.set_title('Inflación Mensual: Original vs. Medias Móviles', fontsize=14, fontweight='bold')
ax.set_xlabel('Fecha')
ax.set_ylabel('Inflación (%)')
ax.legend(loc='upper right', fontsize=10)

plt.tight_layout()
plt.show()

print('🔍 Observa cómo:')
print('   • La media móvil de 3 meses (naranja) sigue de cerca los datos originales')
print('   • La media móvil de 12 meses (roja) muestra la tendencia general sin ruido')
print('   • Ambas tienen un "retraso" natural — reaccionan después de que ocurre el cambio')

In [ ]:
# ============================================================
# Desviación estándar móvil (volatilidad)
# ============================================================

# Volatilidad de la inflación (ventana de 12 meses)
volatilidad_12 = inflacion.rolling(window=12).std()

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Panel superior: serie original
axes[0].plot(inflacion.index, inflacion, color='#2c3e50', linewidth=0.9)
axes[0].set_title('Inflación Mensual', fontweight='bold')
axes[0].set_ylabel('Inflación (%)')

# Panel inferior: volatilidad
axes[1].fill_between(volatilidad_12.index, volatilidad_12, alpha=0.4, color='#e74c3c')
axes[1].plot(volatilidad_12.index, volatilidad_12, color='#c0392b', linewidth=1.5)
axes[1].set_title('Volatilidad (Desv. Estándar Móvil 12 meses)', fontweight='bold')
axes[1].set_ylabel('Desv. Estándar')
axes[1].set_xlabel('Fecha')

plt.suptitle('Inflación y su Volatilidad en el Tiempo', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('🔍 La volatilidad nos dice:')
print('   • Períodos con alta volatilidad = Mayor incertidumbre económica')
print('   • Períodos con baja volatilidad = Mayor estabilidad')
print('   • Esto es útil para planificación presupuestaria y análisis de riesgo')

---

## 📏 5. Estacionariedad y prueba ADF

### ¿Qué es la estacionariedad?

Una serie es **estacionaria** cuando sus propiedades estadísticas (media, varianza) **no cambian con el tiempo.** Imagina una conversación:

- **Serie estacionaria:** "El ruido del ventilador" — siempre suena parecido, sin importar cuándo escuches
- **Serie NO estacionaria:** "El volumen de una sirena acercándose" — cambia sistemáticamente con el tiempo

### ¿Por qué nos importa?

La mayoría de los modelos de pronóstico (incluyendo ARIMA) **requieren datos estacionarios** para funcionar correctamente. Si la serie no es estacionaria, necesitamos transformarla primero — típicamente mediante **diferenciación** (restar cada valor del anterior).

### La prueba de Dickey-Fuller Aumentada (ADF)

Es un test estadístico que nos da una respuesta formal:

- **p-valor < 0.05** → La serie **es estacionaria** (rechazamos la hipótesis de no-estacionariedad)
- **p-valor >= 0.05** → La serie **NO es estacionaria** (no podemos rechazar la hipótesis)

In [ ]:
# ============================================================
# Función auxiliar para la prueba ADF
# ============================================================

def prueba_estacionariedad(serie, nombre='Serie'):
    """
    Aplica la prueba de Dickey-Fuller Aumentada e interpreta los resultados.
    """
    # Eliminar valores nulos
    serie_limpia = serie.dropna()
    
    # Ejecutar la prueba ADF
    resultado = adfuller(serie_limpia, autolag='AIC')
    
    estadistico_adf = resultado[0]
    p_valor = resultado[1]
    valores_criticos = resultado[4]
    
    print(f'=== Prueba ADF para: {nombre} ===')
    print(f'   Estadístico ADF: {estadistico_adf:.4f}')
    print(f'   P-valor:         {p_valor:.6f}')
    print(f'   Valores críticos:')
    for clave, valor in valores_criticos.items():
        print(f'      {clave}: {valor:.4f}')
    
    if p_valor < 0.05:
        print(f'\n   ✅ RESULTADO: La serie ES estacionaria (p={p_valor:.4f} < 0.05)')
    else:
        print(f'\n   ⚠️ RESULTADO: La serie NO es estacionaria (p={p_valor:.4f} >= 0.05)')
        print(f'   → Se recomienda aplicar diferenciación antes de modelar')
    
    return p_valor

In [ ]:
# ============================================================
# Probar estacionariedad de la inflación
# ============================================================

p_original = prueba_estacionariedad(inflacion, 'Inflación mensual (original)')

In [ ]:
# ============================================================
# Si no es estacionaria, aplicamos diferenciación
# ============================================================

# La diferenciación calcula: valor_actual - valor_anterior
# Esto elimina la tendencia y deja solo los cambios período a período
inflacion_diff = inflacion.diff().dropna()  # .diff() = diferencia con el período anterior

# Visualizar original vs. diferenciada
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(inflacion, color='#e74c3c', linewidth=1)
axes[0].set_title('Inflación Mensual — Original', fontweight='bold')
axes[0].set_ylabel('%')

axes[1].plot(inflacion_diff, color='#2980b9', linewidth=1)
axes[1].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[1].set_title('Inflación Mensual — Diferenciada (Δ = cambio mes a mes)', fontweight='bold')
axes[1].set_ylabel('Cambio (%)')
axes[1].set_xlabel('Fecha')

plt.tight_layout()
plt.show()

# Probar estacionariedad de la serie diferenciada
print()
p_diff = prueba_estacionariedad(inflacion_diff, 'Inflación diferenciada (Δ)')

print('\n💡 Nota: La diferenciación elimina la tendencia.')
print('   Si una diferenciación no es suficiente, se puede aplicar dos veces (doble diferenciación).')
print('   Este es el parámetro "d" en ARIMA(p, d, q).')

---

## 🤖 6. Introducción a ARIMA

**ARIMA** es uno de los modelos más usados para pronósticos de series temporales. Su nombre viene de tres componentes:

### ARIMA(p, d, q)

| Parámetro | Nombre | ¿Qué hace? | Analogía |
|-----------|--------|-------------|----------|
| **p** | AutoRegresivo (AR) | Usa valores pasados para predecir el futuro | "Si llovió ayer y anteayer, probablemente llueva hoy" |
| **d** | Integrado (I) | Número de veces que diferenciamos la serie | "En vez de mirar el nivel, miro los cambios" |
| **q** | Media Móvil (MA) | Usa errores de predicción pasados para corregir | "Me equivoqué mucho ayer, así que ajusto mi predicción de hoy" |

### ¿Cómo elegir p, d, q?

Para esta sesión introductoria, usaremos valores comunes:
- **d = 0 o 1** según la prueba ADF
- **p = 1 o 2** (usamos 1-2 valores pasados)
- **q = 1 o 2** (usamos 1-2 errores pasados)

> **Nota:** Existen métodos automáticos para elegir los mejores parámetros (como `auto_arima`), pero eso queda para sesiones más avanzadas. Aquí lo importante es entender **qué hace** el modelo, no optimizarlo al máximo.

In [ ]:
# ============================================================
# ARIMA aplicado al presupuesto ejecutado de la facultad
# ============================================================

# Verificar estacionariedad del presupuesto ejecutado
print('--- Paso 1: Verificar estacionariedad ---')
p_presupuesto = prueba_estacionariedad(ejecutado_mensual, 'Presupuesto ejecutado')

# Determinar el orden de diferenciación
if p_presupuesto < 0.05:
    d = 0  # No necesita diferenciación
    print('\n→ No se necesita diferenciación (d=0)')
else:
    d = 1  # Necesita una diferenciación
    print('\n→ Se aplicará una diferenciación (d=1)')

In [ ]:
# ============================================================
# Ajustar modelo ARIMA
# ============================================================

print('--- Paso 2: Ajustar modelo ARIMA ---')

# Ajustar ARIMA(2, d, 2) — un modelo general razonable
modelo = ARIMA(ejecutado_mensual, order=(2, d, 2))
resultado_arima = modelo.fit()

# Resumen del modelo
print(f'\nModelo: ARIMA(2, {d}, 2)')
print(f'AIC: {resultado_arima.aic:.1f}')  # Menor AIC = mejor modelo
print(f'BIC: {resultado_arima.bic:.1f}')
print(f'\nInterpretación del AIC/BIC:')
print(f'  Estos valores sirven para comparar modelos — el modelo con menor AIC/BIC es preferible.')
print(f'  Por sí solos no indican si el modelo es "bueno" o "malo".')

In [ ]:
# ============================================================
# Pronóstico para los próximos 6 meses
# ============================================================

print('--- Paso 3: Generar pronóstico ---')

# Pronóstico de 6 meses con intervalos de confianza
pronostico = resultado_arima.get_forecast(steps=6)
pred_media = pronostico.predicted_mean
pred_ic = pronostico.conf_int(alpha=0.05)  # Intervalo de confianza al 95%

# Mostrar tabla de pronósticos
tabla_pronostico = pd.DataFrame({
    'Mes': pred_media.index.strftime('%Y-%m'),
    'Pronóstico ($)': pred_media.values,
    'Límite inferior': pred_ic.iloc[:, 0].values,
    'Límite superior': pred_ic.iloc[:, 1].values
})
tabla_pronostico['Pronóstico ($)'] = tabla_pronostico['Pronóstico ($)'].apply(lambda x: f'${x:,.0f}')
tabla_pronostico['Límite inferior'] = tabla_pronostico['Límite inferior'].apply(lambda x: f'${x:,.0f}')
tabla_pronostico['Límite superior'] = tabla_pronostico['Límite superior'].apply(lambda x: f'${x:,.0f}')

print('\nPronóstico de presupuesto ejecutado (próximos 6 meses):')
display(tabla_pronostico)

# Visualizar: datos históricos + pronóstico
fig, ax = plt.subplots(figsize=(14, 6))

# Últimos 24 meses de datos reales
ultimos_24 = ejecutado_mensual[-24:]
ax.plot(ultimos_24.index, ultimos_24.values, color='#2980b9', 
        linewidth=1.5, label='Datos históricos')

# Pronóstico
ax.plot(pred_media.index, pred_media.values, color='#e74c3c', 
        linewidth=2, linestyle='--', label='Pronóstico ARIMA', marker='o', markersize=5)

# Intervalo de confianza
ax.fill_between(pred_ic.index, pred_ic.iloc[:, 0], pred_ic.iloc[:, 1], 
                alpha=0.2, color='#e74c3c', label='Intervalo de confianza 95%')

# Línea divisoria
ax.axvline(x=ejecutado_mensual.index[-1], color='gray', linestyle=':', alpha=0.5)
ax.text(ejecutado_mensual.index[-1], ax.get_ylim()[1]*0.95, '  → Pronóstico', 
        fontsize=10, color='gray')

ax.set_title('Presupuesto Ejecutado: Histórico + Pronóstico ARIMA', fontsize=14, fontweight='bold')
ax.set_xlabel('Fecha')
ax.set_ylabel('Monto ejecutado ($)')
ax.legend(loc='upper left')

plt.tight_layout()
plt.show()

print('\n💡 El intervalo de confianza se amplía hacia el futuro.')
print('   Esto refleja que nuestra incertidumbre CRECE a medida que proyectamos más lejos.')

---

## 🏢 7. Caso aplicado: Proyección de indicadores económicos

Ahora realizaremos el **flujo completo** de análisis de series temporales aplicado a un indicador económico venezolano. Este es el tipo de análisis que podrías realizar en tus clases o investigaciones.

### Flujo de trabajo:
1. Cargar y preparar los datos
2. Visualizar la tendencia
3. Descomponer la serie
4. Evaluar estacionariedad (ADF)
5. Ajustar modelo ARIMA
6. Generar pronóstico con intervalos de confianza

In [ ]:
# ============================================================
# CASO COMPLETO: Proyección de reservas internacionales
# ============================================================

# --- Paso 1: Cargar y preparar ---
print('=' * 60)
print('CASO: Proyección de Reservas Internacionales de Venezuela')
print('=' * 60)

# Recargar datos frescos con fecha como índice
bcv = pd.read_csv(
    '../../datasets/economia/indicadores_bcv.csv',
    parse_dates=['fecha'],
    index_col='fecha'
)

# Seleccionar la serie de reservas internacionales
reservas = bcv['reservas_internacionales_mmusd'].copy()

# Asignar frecuencia mensual
reservas.index = pd.DatetimeIndex(reservas.index, freq='MS')

print(f'\n📊 Datos cargados:')
print(f'   Variable: Reservas Internacionales (millones USD)')
print(f'   Período: {reservas.index[0].strftime("%Y-%m")} a {reservas.index[-1].strftime("%Y-%m")}')
print(f'   Observaciones: {len(reservas)}')
print(f'   Rango: ${reservas.min():.1f} — ${reservas.max():.1f} millones USD')
print(f'   Promedio: ${reservas.mean():.1f} millones USD')

In [ ]:
# --- Paso 2: Visualizar la tendencia ---
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(reservas.index, reservas.values, color='#16a085', linewidth=1.3, label='Reservas')
ax.fill_between(reservas.index, reservas.values, alpha=0.1, color='#16a085')

# Agregar media móvil de 12 meses para ver tendencia
mm12 = reservas.rolling(12).mean()
ax.plot(mm12.index, mm12.values, color='#c0392b', linewidth=2, 
        linestyle='--', label='Media móvil 12 meses')

ax.set_title('Reservas Internacionales de Venezuela', fontsize=14, fontweight='bold')
ax.set_xlabel('Fecha')
ax.set_ylabel('Millones USD')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Paso 3: Descomponer la serie ---
descomp_reservas = seasonal_decompose(reservas, model='additive', period=12)

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)

componentes = [
    (descomp_reservas.observed, 'Serie Original', '#2c3e50'),
    (descomp_reservas.trend, 'Tendencia', '#e74c3c'),
    (descomp_reservas.seasonal, 'Estacionalidad', '#27ae60'),
    (descomp_reservas.resid, 'Residuo', '#8e44ad')
]

for ax, (datos, titulo, color) in zip(axes, componentes):
    ax.plot(datos, color=color, linewidth=1.2)
    ax.set_title(titulo, fontweight='bold', loc='left')
    ax.set_ylabel('MM USD')

axes[-1].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
plt.suptitle('Descomposición: Reservas Internacionales', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# --- Paso 4: Evaluar estacionariedad ---
print('\n📏 EVALUACIÓN DE ESTACIONARIEDAD')
print('-' * 40)
p_reservas = prueba_estacionariedad(reservas, 'Reservas internacionales')

# Si no es estacionaria, probar con diferenciación
if p_reservas >= 0.05:
    print('\n→ Aplicando diferenciación...')
    reservas_diff = reservas.diff().dropna()
    p_reservas_diff = prueba_estacionariedad(reservas_diff, 'Reservas diferenciadas')
    d_reservas = 1
else:
    d_reservas = 0

In [ ]:
# --- Paso 5: Ajustar modelo ARIMA ---
print('\n🤖 AJUSTE DEL MODELO ARIMA')
print('-' * 40)

# Probar varios modelos y quedarnos con el de menor AIC
mejor_aic = float('inf')
mejor_orden = None
mejor_modelo = None

ordenes_a_probar = [
    (1, d_reservas, 1),
    (2, d_reservas, 1),
    (1, d_reservas, 2),
    (2, d_reservas, 2),
    (3, d_reservas, 1),
]

print(f'\nProbando {len(ordenes_a_probar)} configuraciones de ARIMA...')
for orden in ordenes_a_probar:
    try:
        mod = ARIMA(reservas, order=orden).fit()
        print(f'   ARIMA{orden} → AIC = {mod.aic:.1f}')
        if mod.aic < mejor_aic:
            mejor_aic = mod.aic
            mejor_orden = orden
            mejor_modelo = mod
    except Exception as e:
        print(f'   ARIMA{orden} → Error: {str(e)[:50]}')

print(f'\n✅ Mejor modelo: ARIMA{mejor_orden} (AIC = {mejor_aic:.1f})')

In [ ]:
# --- Paso 6: Pronóstico de 12 meses ---
print('\n📈 PRONÓSTICO A 12 MESES')
print('-' * 40)

# Generar pronóstico
forecast = mejor_modelo.get_forecast(steps=12)
fc_media = forecast.predicted_mean
fc_ci = forecast.conf_int(alpha=0.05)

# Tabla de resultados
tabla_fc = pd.DataFrame({
    'Mes': fc_media.index.strftime('%Y-%m'),
    'Pronóstico (MM USD)': fc_media.values.round(1),
    'Límite inferior': fc_ci.iloc[:, 0].values.round(1),
    'Límite superior': fc_ci.iloc[:, 1].values.round(1)
})
display(tabla_fc)

# Gráfico final
fig, ax = plt.subplots(figsize=(14, 6))

# Serie histórica completa
ax.plot(reservas.index, reservas.values, color='#16a085', 
        linewidth=1.3, label='Datos históricos')

# Pronóstico
ax.plot(fc_media.index, fc_media.values, color='#e74c3c', 
        linewidth=2.5, linestyle='--', marker='o', markersize=4, 
        label=f'Pronóstico ARIMA{mejor_orden}')

# Intervalo de confianza al 95%
ax.fill_between(fc_ci.index, fc_ci.iloc[:, 0], fc_ci.iloc[:, 1], 
                alpha=0.2, color='#e74c3c', label='IC 95%')

# Línea divisoria
ax.axvline(x=reservas.index[-1], color='gray', linestyle=':', alpha=0.6)

ax.set_title('Reservas Internacionales: Histórico + Pronóstico ARIMA (12 meses)', 
             fontsize=14, fontweight='bold')
ax.set_xlabel('Fecha')
ax.set_ylabel('Millones USD')
ax.legend(loc='best')

plt.tight_layout()
plt.show()

print('\n🔍 Interpretación del pronóstico:')
print(f'   • El modelo proyecta reservas promedio de ${fc_media.mean():.1f} MM USD para los próximos 12 meses.')
print(f'   • El rango de incertidumbre va de ${fc_ci.iloc[:, 0].min():.1f} a ${fc_ci.iloc[:, 1].max():.1f} MM USD.')
print(f'   • A mayor horizonte de pronóstico, mayor incertidumbre (bandas más anchas).')
print(f'\n⚠️ IMPORTANTE: Un modelo estadístico NO puede predecir eventos disruptivos')
print(f'   (cambios de política, crisis, reformas). El pronóstico asume que las condiciones')
print(f'   históricas se mantienen. Siempre acompañar con análisis cualitativo.')

---

## ✏️ Ejercicios

Pon en práctica lo aprendido con estos ejercicios usando los datos reales del BCV y de la facultad.

### Ejercicio 1: Descomponer el tipo de cambio

Usando el dataset `indicadores_bcv.csv`, descompón la serie `tipo_cambio_indice` en sus componentes (tendencia, estacionalidad, residuo) e interpreta cada uno:

- ¿Cuál es la tendencia general del tipo de cambio en el período 2015-2025?
- ¿Existe un patrón estacional? ¿En qué meses tiende a ser más alto o bajo?
- ¿Los residuos muestran algún evento atípico?

In [ ]:
# ============================================================
# EJERCICIO 1: Descomposición del tipo de cambio
# ============================================================

# Paso 1: Cargar la serie de tipo de cambio
# Tu código aquí


# Paso 2: Aplicar seasonal_decompose con period=12
# Tu código aquí


# Paso 3: Graficar los 4 componentes
# Tu código aquí


# Paso 4: Escribir tu interpretación como comentarios
# Tendencia: ...
# Estacionalidad: ...
# Residuos: ...


### Ejercicio 2: Pronóstico del presupuesto de Personal

Usando el dataset `presupuesto_facultad.csv`, filtra solo la partida **"Personal"**, agrega por mes, y genera un pronóstico ARIMA para los próximos 12 meses:

1. Filtra `presupuesto[presupuesto['partida'] == 'Personal']`
2. Agrega el `ejecutado` por fecha
3. Verifica estacionariedad con la prueba ADF
4. Ajusta un modelo ARIMA
5. Genera el pronóstico y grafícalo con intervalos de confianza

In [ ]:
# ============================================================
# EJERCICIO 2: Pronóstico ARIMA para presupuesto de Personal
# ============================================================

# Paso 1: Filtrar partida 'Personal'
# Tu código aquí


# Paso 2: Crear fecha y agregar por mes
# Tu código aquí


# Paso 3: Prueba ADF
# Tu código aquí


# Paso 4: Ajustar ARIMA
# Tu código aquí


# Paso 5: Pronóstico de 12 meses con gráfico
# Tu código aquí


### Ejercicio 3: Medias móviles de reservas internacionales

Calcula y grafica la media móvil de **6 meses** para la serie `reservas_internacionales_mmusd` del dataset `indicadores_bcv.csv`:

1. Grafica la serie original y la media móvil de 6 meses superpuestas
2. Identifica los períodos de mayor caída de reservas
3. Calcula también la desviación estándar móvil de 6 meses y grafícala debajo

In [ ]:
# ============================================================
# EJERCICIO 3: Medias móviles de reservas internacionales
# ============================================================

# Paso 1: Seleccionar la serie de reservas
# Tu código aquí


# Paso 2: Calcular media móvil de 6 meses
# Tu código aquí


# Paso 3: Calcular desviación estándar móvil de 6 meses
# Tu código aquí


# Paso 4: Crear gráfico con 2 paneles (serie + mm arriba, volatilidad abajo)
# Tu código aquí


# Paso 5: Identificar y comentar períodos de mayor caída
# Tu código aquí


---

## 📋 Resumen

En esta sesión aprendimos:

| Concepto | Lo esencial |
|----------|-------------|
| **Serie temporal** | Datos medidos a intervalos regulares en el tiempo. El orden importa. |
| **Componentes** | Tendencia + Estacionalidad + Ciclo + Residuo |
| **Descomposición** | `seasonal_decompose()` separa automáticamente los componentes |
| **Media móvil** | Suaviza la serie para ver tendencias. Ventana corta = detalle, larga = tendencia |
| **Estacionariedad** | Propiedad necesaria para modelar. Se evalúa con la prueba ADF (p < 0.05 = estacionaria) |
| **Diferenciación** | Transforma una serie no estacionaria restando valores consecutivos |
| **ARIMA(p,d,q)** | Modelo de pronóstico: AR (valores pasados) + I (diferenciación) + MA (errores pasados) |
| **Intervalos de confianza** | La incertidumbre crece con el horizonte de pronóstico |

### Flujo de trabajo recomendado:

```
Cargar datos → Visualizar → Descomponer → Prueba ADF → 
Diferenciar (si necesario) → Ajustar ARIMA → Pronosticar → Interpretar
```

### Limitaciones importantes:

- Los modelos ARIMA asumen que el futuro se comportará como el pasado
- No pueden predecir eventos disruptivos (crisis, cambios de política)
- Funcionan mejor con series largas y estables
- Siempre deben complementarse con análisis cualitativo y juicio experto

## 📚 Referencias

1. **statsmodels — Documentación oficial de series temporales:**  
   https://www.statsmodels.org/stable/tsa.html

2. **Hyndman, R.J. & Athanasopoulos, G. — *Forecasting: Principles and Practice* (3ra ed.):**  
   https://otexts.com/fpp3/ (Libro gratuito en línea, referencia mundial en pronósticos)

3. **Pandas — Funcionalidades de series temporales:**  
   https://pandas.pydata.org/docs/user_guide/timeseries.html

4. **Banco Central de Venezuela — Indicadores económicos:**  
   https://www.bcv.org.ve/estadisticas

5. **Introducción a ARIMA (en español):**  
   https://www.cienciadedatos.net/documentos/py51-modelos-arima-python.html